# Tries (Prefix Trees)

Notes and runnable examples on the **trie** — a tree keyed by the *characters* of its keys, so strings that share a prefix share a path. It's the structure behind autocomplete, spell-checkers, and IP routing. Like the other from-scratch structures here, Python ships none — and the honest question is **when a trie actually beats a plain `set`**.

**Contents**

1. **What it is** — paths of characters, shared prefixes
2. **A trie from scratch**
3. **The prefix superpower** — autocomplete & shared nodes
4. **The reality check** — memory, and when a `set` wins
5. **When to use what**
6. **Complexity cheat-sheet**

## 1. What it is

A **trie** (a.k.a. *prefix tree*; the name comes from re**trie**val) stores a set of strings as **paths of characters** from a root. Each node holds a map of `character -> child node`, plus a flag marking whether a complete word ends there. To look a key up, you spell it out one character at a time, walking down the tree.

Two consequences define it:

- **Shared prefixes share nodes** — `car`, `card`, and `care` all reuse the `c -> a -> r` path. The structure *is* the set of prefixes.
- **Operations cost O(L)** in the key length L — *independent of how many keys the trie holds*. A million-word trie still finds a 5-letter word in 5 steps.

And the flag matters: a path can exist as a **prefix** without being a stored **word** — you can walk `c -> a` even if `"ca"` was never inserted.

## 2. A trie from scratch

A node is just a `dict` of children plus an `is_word` flag. `insert` walks (creating nodes as needed) character by character; `search` and `starts_with` walk and then either check the flag or simply confirm the path exists. `dict.setdefault` does create-or-reuse-child in one step.

In [1]:
class TrieNode:
    __slots__ = ("children", "is_word")
    def __init__(self):
        self.children = {}            # char -> TrieNode
        self.is_word = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):                       # O(len(word))
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_word = True

    def _find(self, s):                           # walk the path, return the node or None
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    def __contains__(self, word):                 # exact: path exists AND ends a word
        node = self._find(word)
        return node is not None and node.is_word

    def starts_with(self, prefix):                # is there ANY word with this prefix?
        return self._find(prefix) is not None

trie = Trie()
for w in ["cat", "car", "card", "care", "dog", "do"]:
    trie.insert(w)

print("'car' in trie     :", "car" in trie)            # True
print("'ca'  in trie     :", "ca" in trie)             # False - a prefix, never inserted as a word
print("starts_with('ca') :", trie.starts_with("ca"))   # True
print("starts_with('xy') :", trie.starts_with("xy"))   # False


'car' in trie     : True
'ca'  in trie     : False
starts_with('ca') : True
starts_with('xy') : False


## 3. The prefix superpower — autocomplete & shared nodes

This is the thing a `set` *can't* do cheaply. To list **every word starting with a prefix**, walk to the prefix's node once (O(L)), then DFS the subtree beneath it — you only ever touch nodes that genuinely share the prefix. That's autocomplete.

In [2]:
def with_prefix(trie, prefix):
    node = trie._find(prefix)
    out = []
    if node is None:
        return out
    def dfs(node, suffix):
        if node.is_word:
            out.append(prefix + suffix)
        for ch in sorted(node.children):          # sorted children -> alphabetical results
            dfs(node.children[ch], suffix + ch)
    dfs(node, "")
    return out

print("autocomplete 'car':", with_prefix(trie, "car"))   # ['car', 'card', 'care']
print("autocomplete 'do' :", with_prefix(trie, "do"))    # ['do', 'dog']

# shared prefixes mean far fewer nodes than total characters:
def count_nodes(trie):
    stack, n = [trie.root], 0
    while stack:
        node = stack.pop(); n += 1
        stack.extend(node.children.values())
    return n

words = ["cat", "car", "card", "care", "dog", "do"]
print("total characters:", sum(len(w) for w in words))           # 19
print("trie nodes      :", count_nodes(trie) - 1, "(excl. root)")  # 9 - prefixes collapse


autocomplete 'car': ['car', 'card', 'care']
autocomplete 'do' : ['do', 'dog']
total characters: 19
trie nodes      : 9 (excl. root)


A free bonus falls out of recursing children in sorted order: autocomplete results come back **alphabetically sorted** with no extra work — another thing a hash set simply can't offer.

## 4. The reality check — memory, and when a `set` wins

Tries are easy to oversell. In Python every node is an object **and** carries a `dict` of children, so per-node overhead is large — roughly 100+ bytes for a node with a single child.

In [3]:
import sys

node = TrieNode()
print("one TrieNode (with __slots__):", sys.getsizeof(node), "bytes")
print("+ its children dict          :", sys.getsizeof(node.children), "bytes")
print("=> ~100+ bytes per node, one node per distinct prefix character")

words = {"cat", "car", "card", "care", "dog", "do"}
print("a set of the same 6 words    :", sys.getsizeof(words), "bytes (container) + the strings")


one TrieNode (with __slots__): 48 bytes
+ its children dict          : 64 bytes
=> ~100+ bytes per node, one node per distinct prefix character
a set of the same 6 words    : 472 bytes (container) + the strings


So for **plain membership** ("is this exact word present?"), a `set` is simpler, faster, and far lighter — hash the whole string once and you're done; don't reach for a trie. A trie earns its keep only for what a hash *can't* do:

- **prefix queries** / autocomplete (`starts_with`, "all words with prefix X"),
- **ordered** iteration and **longest-prefix matching** (IP routing tables, dictionaries),
- heavily overlapping keys where shared prefixes pay off.

Real systems use **compressed** variants — a *radix tree* / PATRICIA trie collapses single-child chains into one edge to cut that node overhead. (A `str`-keyed `dict` already nails exact lookup; the trie is specifically about the *prefix* dimension.)

## 5. When to use what

| You need... | Reach for | Why |
|---|---|---|
| Exact membership / key lookup | `set` / `dict` | hashes the key once; tiny and fast, but no prefix info |
| Autocomplete / "all keys with prefix X" | trie | walk to the prefix, then DFS the subtree |
| Keys returned in sorted order | trie (or `sortedcontainers`) | sorted children ⇒ sorted output |
| Longest-prefix match (routing, dictionaries) | trie / radix tree | matches as it descends the path |
| Many keys, heavy shared prefixes, tight memory | radix / PATRICIA trie | collapses single-child chains |

## 6. Complexity cheat-sheet

<sub>L = key length, n = number of keys, k = matches for a prefix.</sub>

| Operation | Trie | `set` / `dict` |
|---|:---:|:---:|
| Insert key | O(L) | O(L)† |
| Exact lookup | O(L) | O(L)† |
| Delete key | O(L) | O(L)† |
| `starts_with(prefix)` | O(L) | — (unsupported) |
| All keys with a prefix | O(L + k) | O(n) full scan |
| Sorted iteration | O(total chars) | O(n log n) |
| Memory | high (a node + a dict each) | low (one entry per key) |

<sub>† hashing or comparing the key is itself O(L) — the usual "O(1)" for a hash hides the key-length factor. A trie's edge isn't raw lookup speed; it's the **prefix and ordering** operations a hash fundamentally can't do.</sub>